In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

print(f"Scikit-learn version: {sklearn.__version__}")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.metrics import (mean_squared_error, mean_absolute_error, 
                           root_mean_squared_error, accuracy_score,
                           confusion_matrix, precision_score, recall_score,
                           f1_score, classification_report)

print(" Все библиотеки загружены!")

Scikit-learn version: 1.8.0
 Все библиотеки загружены!


In [5]:
df = pd.read_csv('processed_titanic-checkpoint.csv')

In [7]:
feature_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
if 'Sex_male' in df.columns:
    feature_cols.append('Sex_male')
if 'Embarked_Q' in df.columns:
    feature_cols.append('Embarked_Q')
if 'Embarked_S' in df.columns:
    feature_cols.append('Embarked_S')

available_cols = [col for col in feature_cols if col in df.columns]
X = df[available_cols]
y_reg = df['Fare']
y_clf = df['Survived']

In [9]:
for col in X.columns:
    if X[col].isnull().any():
        median_val = X[col].median()
        X[col] = X[col].fillna(median_val)
        print(f"Заполнены пропуски в {col} медианой {median_val}")

print("\nПроверка после заполнения:")
print(X.isnull().sum())


Проверка после заполнения:
Pclass    0
Age       0
SibSp     0
Parch     0
Fare      0
dtype: int64


In [11]:
X_train_reg, X_temp_reg, y_train_reg, y_temp_reg = train_test_split(
    X, y_reg, test_size=0.4, random_state=42
)
X_test_reg, X_val_reg, y_test_reg, y_val_reg = train_test_split(
    X_temp_reg, y_temp_reg, test_size=0.5, random_state=42
)

X_train_clf, X_temp_clf, y_train_clf, y_temp_clf = train_test_split(
    X, y_clf, test_size=0.4, random_state=42
)
X_test_clf, X_val_clf, y_test_clf, y_val_clf = train_test_split(
    X_temp_clf, y_temp_clf, test_size=0.5, random_state=42
)

print(" данные разделены")
print(f"Обучающая выборка: {X_train_reg.shape}")
print(f"Тестовая выборка: {X_test_reg.shape}")
print(f"Валидационная выборка: {X_val_reg.shape}")

 данные разделены
Обучающая выборка: (534, 5)
Тестовая выборка: (178, 5)
Валидационная выборка: (179, 5)


In [13]:
scaler = StandardScaler()
X_train_reg_scaled = scaler.fit_transform(X_train_reg)
X_test_reg_scaled = scaler.transform(X_test_reg)
X_train_clf_scaled = scaler.fit_transform(X_train_clf)
X_test_clf_scaled = scaler.transform(X_test_clf)

print("Масштабирование выполнено")
print(f"Пропусков после масштабирования: {np.isnan(X_train_reg_scaled).sum()}")

Масштабирование выполнено
Пропусков после масштабирования: 0


In [14]:
lr_model = LinearRegression()
lr_model.fit(X_train_reg_scaled, y_train_reg)
y_pred_reg = lr_model.predict(X_test_reg_scaled)

print("Линейная регрессия построена")
print(f"RMSE: {root_mean_squared_error(y_test_reg, y_pred_reg):.4f}")

Линейная регрессия построена
RMSE: 0.0000


In [16]:
print("Оценка регрессионной модели")

print(f"MSE: {mean_squared_error(y_test_reg, y_pred_reg):.4f}")
print(f"RMSE: {root_mean_squared_error(y_test_reg, y_pred_reg):.4f}")
print(f"MAE: {mean_absolute_error(y_test_reg, y_pred_reg):.4f}")

print("\nУлучшение модели с помощью Ridge регуляризации:")
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_reg_scaled, y_train_reg)
y_pred_ridge = ridge.predict(X_test_reg_scaled)
print(f"Ridge RMSE: {root_mean_squared_error(y_test_reg, y_pred_ridge):.4f}")

Оценка регрессионной модели
MSE: 0.0000
RMSE: 0.0000
MAE: 0.0000

Улучшение модели с помощью Ridge регуляризации:
Ridge RMSE: 0.0815


In [17]:
print(" КЛАССИФИКАЦИЯ")

logreg = LogisticRegression(random_state=42, max_iter=1000)
logreg.fit(X_train_clf_scaled, y_train_clf)
y_pred_clf = logreg.predict(X_test_clf_scaled)

print("Логистическая регрессия построена")

 КЛАССИФИКАЦИЯ
Логистическая регрессия построена


In [19]:
print(" Оценка классификационной модели")

print(f"Accuracy: {accuracy_score(y_test_clf, y_pred_clf):.4f}")
print(f"Precision: {precision_score(y_test_clf, y_pred_clf):.4f}")
print(f"Recall: {recall_score(y_test_clf, y_pred_clf):.4f}")
print(f"F1-score: {f1_score(y_test_clf, y_pred_clf):.4f}")

 Оценка классификационной модели
Accuracy: 0.6910
Precision: 0.6250
Recall: 0.3846
F1-score: 0.4762


In [20]:
cm = confusion_matrix(y_test_clf, y_pred_clf)
print("\nМатрица ошибок:")
print(f"[[{cm[0,0]} {cm[0,1]}]")
print(f" [{cm[1,0]} {cm[1,1]}]]")


Матрица ошибок:
[[98 15]
 [40 25]]


In [21]:
print("\n подбор параметра C")
for C in [0.1, 1.0, 10.0]:
    model = LogisticRegression(C=C, random_state=42, max_iter=1000)
    model.fit(X_train_clf_scaled, y_train_clf)
    y_pred = model.predict(X_test_clf_scaled)
    print(f"C={C}: F1={f1_score(y_test_clf, y_pred):.4f}")


 подбор параметра C
C=0.1: F1=0.4854
C=1.0: F1=0.4762
C=10.0: F1=0.4717


In [22]:
with open('results_lab2.txt', 'w', encoding='utf-8') as f:
    f.write(f"Регрессия (Fare): RMSE={root_mean_squared_error(y_test_reg, y_pred_reg):.4f}\n")
    f.write(f"Классификация (Survived): Accuracy={accuracy_score(y_test_clf, y_pred_clf):.4f}\n")